In [6]:
library(data.table)
library(qvalue)

In [7]:
saige_dir = '/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/eqtl_results/saige_qtl/december24_freeze/'

In [8]:
output_dir_all = '/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/ms_tables/rare_variant_gene_level_results/'
output_dir_functional = '/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/ms_tables/rare_variant_gene_level_results_functional_annotations/'

In [9]:
celltypes = list.files(saige_dir)
celltypes = celltypes[!(celltypes %in% c('README.md','CD4_TCM_sample_perm0','annotate','coloc_results'))]
length(celltypes)
celltypes

[1] 28

[1] "ASDC"              "B_intermediate"    "B_memory"         
 [4] "B_naive"           "CD14_Mono"         "CD16_Mono"        
 [7] "CD4_CTL"           "CD4_Naive"         "CD4_Proliferating"
[10] "CD4_TCM"           "CD4_TEM"           "CD8_Naive"        
[13] "CD8_Proliferating" "CD8_TCM"           "CD8_TEM"          
[16] "cDC1"              "cDC2"              "dnT"              
[19] "gdT"               "HSPC"              "ILC"              
[22] "MAIT"              "NK"                "NK_CD56bright"    
[25] "NK_Proliferating"  "pDC"               "Plasmablast"      
[28] "Treg"

In [10]:
#Code adpated from the STAR package https://github.com/xihaoli/STAAR/blob/dc4f7e509f4fa2fb8594de48662bbd06a163108c/R/CCT.R wtih a modifitcaiton: when indiviudal p-value = 1, use minimum p-value 
#' An analytical p-value combination method using the Cauchy distribution
#'
#' The \code{CCT} function takes in a numeric vector of p-values, a numeric
#' vector of non-negative weights, and return the aggregated p-value using Cauchy method.
#' @param pvals a numeric vector of p-values, where each of the element is
#' between 0 to 1, to be combined.
#' @param weights a numeric vector of non-negative weights. If \code{NULL}, the
#' equal weights are assumed.
#' @return the aggregated p-value combining p-values from the vector \code{pvals}.
#' @examples pvalues <- c(2e-02,4e-04,0.2,0.1,0.8)
#' @examples CCT(pvals=pvalues)
#' @references Liu, Y., & Xie, J. (2020). Cauchy combination test: a powerful test
#' with analytic p-value calculation under arbitrary dependency structures.
#' \emph{Journal of the American Statistical Association 115}(529), 393-402.
#' (\href{https://www.tandfonline.com/doi/full/10.1080/01621459.2018.1554485}{pub})
#' @export

CCT <- function(pvals, weights=NULL){
  #### check if there is NA
  if(sum(is.na(pvals)) > 0){
    stop("Cannot have NAs in the p-values!")
  }

  #### check if all p-values are between 0 and 1
  if((sum(pvals<0) + sum(pvals>1)) > 0){
    stop("All p-values must be between 0 and 1!")
  }

  #### check if there are p-values that are either exactly 0 or 1.
  is.zero <- (sum(pvals==0)>=1)
  is.one <- (sum(pvals==1)>=1)
  #if(is.zero && is.one){
  #  stop("Cannot have both 0 and 1 p-values!")
  #}
  if(is.zero){
    return(0)
  }
  if(is.one){
    #warning("There are p-values that are exactly 1!")
    return(min(1,(min(pvals))*(length(pvals))))
  }

  #### check the validity of weights (default: equal weights) and standardize them.
  if(is.null(weights)){
    weights <- rep(1/length(pvals),length(pvals))
  }else if(length(weights)!=length(pvals)){
    stop("The length of weights should be the same as that of the p-values!")
  }else if(sum(weights < 0) > 0){
    stop("All the weights must be positive!")
  }else{
    weights <- weights/sum(weights)
  }

  #### check if there are very small non-zero p-values
  is.small <- (pvals < 1e-16)
  if (sum(is.small) == 0){
    cct.stat <- sum(weights*tan((0.5-pvals)*pi))
  }else{
    cct.stat <- sum((weights[is.small]/pvals[is.small])/pi)
    cct.stat <- cct.stat + sum(weights[!is.small]*tan((0.5-pvals[!is.small])*pi))
  }

  #### check if the test statistic is very large.
  if(cct.stat > 1e+15){
    pval <- (1/cct.stat)/pi
  }else{
    pval <- 1-pcauchy(cct.stat)
  }
  return(pval)
}

In [11]:
get_CCT_pvalue = function(pvalue, weights=NULL){
   pvals = pvalue
   notna = which(!is.na(pvals))
   if(length(notna) > 0){
     pvals = pvals[!is.na(pvals)]
     cctpval = CCT(pvals, weights=weights)
   }else{
     cctpval = NA
   }
   return(cctpval)
}

In [ ]:
# all variants

In [19]:
for (celltype in celltypes){
    rare_set_file = paste0(saige_dir,celltype,'/',celltype,'_all_cis_rv_set_test_results.tsv')
    rare_set_df = as.data.frame(fread(rare_set_file))
    for (i in 1:nrow(rare_set_df)){
            rare_set_df[i,'Pvalue_combined_Burden_SKAT'] = get_CCT_pvalue(c(rare_set_df[i,'Pvalue_Burden'], rare_set_df[i,'Pvalue_SKAT']))
        }
    rare_set_df$celltype = celltype
    rare_set_df$Pvalue = c()
    rare_set_df$Pvalue_ACATV = c()
    rare_set_df$Pvalue_SKATO = c()
    rare_set_df <- rare_set_df[, c(14,1:5,13, 6:12)]
    fwrite(rare_set_df,paste0(output_dir_all, celltype,'_rare_variant_gene_level_results.tsv'))
}

In [20]:
# functional variants

In [21]:
anno_folder = '/directflow/SCCGGroupShare/projects/anncuo/TenK10K_pilot/tenk10k/review_files/rv_anno_test_results/functional_OR_open/'

In [25]:
for (celltype in celltypes){
    if (celltype %in% c('CD8_Proliferating','ILC')){next}
    rare_set_file = paste0(anno_folder,celltype,'_all_cis_rv_set_test_results.tsv')
    rare_set_df = as.data.frame(fread(rare_set_file))
    for (i in 1:nrow(rare_set_df)){
            rare_set_df[i,'Pvalue_combined_Burden_SKAT'] = get_CCT_pvalue(c(rare_set_df[i,'Pvalue_Burden'], rare_set_df[i,'Pvalue_SKAT']))
        }
    rare_set_df$Region = gsub('_.*','',rare_set_df$Region)
    rare_set_df$celltype = celltype
    rare_set_df$Pvalue = c()
    rare_set_df$Pvalue_ACATV = c()
    rare_set_df$Pvalue_SKATO = c()
    rare_set_df <- rare_set_df[, c(14,1:5,13, 6:12)]
    fwrite(rare_set_df,paste0(output_dir_functional, celltype,'_rare_variant_gene_level_results_annos.tsv'))
}